In [4]:
import deeplake
import time
import base64

schema = {
    "id": deeplake.types.UInt64(),
    "embedding": deeplake.types.Embedding(768),
    "caption": deeplake.types.Text(),
    "base64encoding": deeplake.types.Text(),
    "streamId": deeplake.types.Text(),
}

path = "file://database"
df = 0
try:
    ds = deeplake.open(path)
except Exception as e:
    print(f"Failed to open dataset: {e}")
    ds = deeplake.create(path, schema=schema)




# Add Column to Dataset

In [7]:
import deeplake
import os

path = 'al://second-sight/video-recordings'

try:
    ds = deeplake.open(url = path, token = os.getenv("ACTIVELOOP_TOKEN"))
    ds.add_column("frames", deeplake.types.Sequence(deeplake.types.Image(sample_compression="jpeg")
))
except Exception as e:
    print(f"Failed to open dataset: {e}")

In [ ]:
import deeplake
import time
import os
import cv2
import sys
from dotenv import load_dotenv

caption, embedding = None, None

load_dotenv()
# Add the directory containing GoogleGemini.py to the Python path
# '.' represents the current directory, assuming the notebook is in the same folder as the .py file
module_path = '.' 
if module_path not in sys.path:
    sys.path.append(module_path)

# Now you can import the functions
try:
    from GoogleGemini import generate_video_caption, generate_text_embedding
    print("Successfully imported functions from GoogleGemini.py")
except ImportError as e:
    print(f"Failed to import functions: {e}")
except Exception as e:
    print(f"An error occurred during import: {e}")

# Define your schema
schema = {
    "id": deeplake.types.UInt64(),  # Unique identifier for each entry (e.g., session ID or video clip ID)
    "frames": deeplake.types.Sequence(deeplake.types.Image(sample_compression="jpeg")),  # Video frames as a sequence of images
    "captions": deeplake.types.Text(),  # Single caption for the entire video
    "embeddings": deeplake.types.Embedding(768),  # Embedding for the entire video
}

path = 'al://second-sight/testing'

try:
    ds = deeplake.open(path, token = os.getenv("ACTIVELOOP_TOKEN"))
except Exception as e:
    print(f"Failed to open dataset: {e}")
    ds = deeplake.create(url = path, schema=schema, token= os.getenv("ACTIVELOOP_TOKEN"))

# Function to extract frames and motion boxes from video
def extract_frames_and_motion_boxes(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    motion_boxes = []
    previous_frame = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (21, 21), 0)
        current_time = time.time()

        if previous_frame is None:
            previous_frame = gray
        else:
            # Calculate difference between consecutive frames
            delta = cv2.absdiff(previous_frame, gray)
            thresh = cv2.threshold(delta, 50, 255, cv2.THRESH_BINARY)[1]
            thresh = cv2.dilate(thresh, None, iterations=2)
            contours, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            # Detect motion and get bounding boxes
            for c in contours:
                if cv2.contourArea(c) < 1500:  # Ignore small contours
                    continue
                x, y, w, h = cv2.boundingRect(c)
                motion_boxes.append([x, y, w, h])  # Store the bounding box (x, y, width, height)

        frames.append(frame)  # Add frame to list
        previous_frame = gray  # Update previous frame

    cap.release()
    return frames, motion_boxes

# Function to process the video, generate caption, embedding, and upload to DeepLake
async def process_and_upload_video(video_path, dataset):
    if not os.path.exists(video_path):
        print(f"Error: Video file not found at {video_path}")
        return

    print(f"Processing video: {video_path}")
    
    try:
        # 1. Extract Frames and Motion Boxes
        print("Extracting frames and motion boxes...")
        frames, motion_boxes = extract_frames_and_motion_boxes(video_path)
        print(f"Extracted {len(frames)} frames and {len(motion_boxes)} motion boxes.")

        # 2. Generate Caption
        print("Generating caption...")
        caption = generate_video_caption(video_path)
        print(f"Generated Caption: {caption}")

        # 3. Generate Embedding from Caption
        print("Generating embedding...")
        embedding = generate_text_embedding(caption)
        print(f"Generated Embedding (first 5 dims): {embedding[:5]}")
        print(f"Embedding length: {len(embedding)}")
        repeated_captions = [caption] * len(frames)  # Repeat the caption for each frame

        # 4. Prepare data for Deep Lake
        data_to_upload = {
            'id': [int(time.time() * 1000)],  # Unique ID based on timestamp
            'frames': [frames],  # Encode frames to JPEG
            'captions': [caption],  # Single caption for the entire video (single text)
            'embeddings': [embedding] ,  # Single embedding for the entire video
        }

        # 5. Append to Deep Lake dataset
        print("Appending data to Deep Lake...")
        dataset.append(data_to_upload)
        dataset.commit()
        print("Data successfully appended to Deep Lake.")

    except FileNotFoundError:
        print(f"Error: The video file '{video_path}' was not found during processing.")
    except Exception as e:
        print(f"An error occurred during processing or upload: {e}")

# Execute the async function
async def main():
    video_file_path = "tempVideo/testdelivery.mov"  # Path to your test video
    print("Starting video processing and upload...")
    await process_and_upload_video(video_file_path, ds)
    print("Process finished.")

# For Jupyter environments, top-level await is allowed
# In other environments, use `asyncio.run(main())` instead
await main()


Successfully imported functions from GoogleGemini.py
Starting video processing and upload...
Processing video: tempVideo/testdelivery.mov
Extracting frames and motion boxes...
Extracted 903 frames and 695 motion boxes.
Generating caption...
Uploading video...
Completed upload: https://generativelanguage.googleapis.com/v1beta/files/ab1sbaxhqyaz
...................Generated Caption: Here's a breakdown of the video:

1.  **Main Action:** The central event in the video shows a delivery person who has just dropped off an Amazon package. They then decide to spray some sort of liquid (likely a disinfectant or cleaner) onto the area around the package.

2.  **Setting and Environment:** The scene is set outside a modern home. The backdrop features a black front door with a decorative glass panel. The door is framed by white, textured stone walls, and a welcome mat sits at the entrance. There's a tall, shiny silver vase with greenery to the right of the door. The overall environment suggests a w

# Convert Video to Frames and Back

In [30]:
video_path = "tempVideo/testdelivery.mov"  # Path to your test video
frames, boxes = extract_frames_and_motion_boxes(video_path)

# Convert frames back to video
output_path = "recreated_video.mp4"

# --- Function to Create Video from Frames ---
def create_video_from_frames(frames, output_path, fps=30):
    """Creates a video file from a list of frames."""
    if not frames:
        print("Error: No frames provided to create video.")
        return False

    # Get frame dimensions from the first frame
    height, width, layers = frames[0].shape
    size = (width, height)

    # Define the codec and create VideoWriter object
    # Common codecs: 'mp4v' for .mp4, 'XVID' for .avi
    # Adjust fourcc based on desired output format and available codecs
    fourcc = cv2.VideoWriter_fourcc(*'mp4v') # For .mp4 output
    out = cv2.VideoWriter(output_path, fourcc, fps, size)

    if not out.isOpened():
        print(f"Error: Could not open VideoWriter for path {output_path}")
        # Try a different codec if the first fails (optional)
        # fourcc = cv2.VideoWriter_fourcc(*'XVID') # For .avi
        # out = cv2.VideoWriter(output_path.replace('.mp4', '.avi'), fourcc, fps, size)
        # if not out.isOpened():
        #     print(f"Error: Still could not open VideoWriter with alternative codec.")
        #     return False
        return False


    print(f"Writing {len(frames)} frames to {output_path} at {fps} FPS...")
    for frame in frames:
        # Ensure frame dimensions match if they vary (though they shouldn't from extract_frames)
        if frame.shape[0] != height or frame.shape[1] != width:
            print(f"Warning: Frame size mismatch ({frame.shape[:2]}) vs expected ({height, width}). Resizing.")
            frame = cv2.resize(frame, size)
        out.write(frame) # Write the frame

    out.release() # Release the VideoWriter
    print(f"Successfully created video: {output_path}")
    return True

# --- Call the Function ---
if create_video_from_frames(frames, output_path):
    print(f"Video created successfully at {output_path}")

Writing 903 frames to recreated_video.mp4 at 30 FPS...
Successfully created video: recreated_video.mp4
Video created successfully at recreated_video.mp4


# Add Data Dynamically

In [2]:
import deeplake
import time
import os
import cv2
import sys
from dotenv import load_dotenv

caption, embedding = None, None

load_dotenv()
# Add the directory containing GoogleGemini.py to the Python path
# '.' represents the current directory, assuming the notebook is in the same folder as the .py file
module_path = '.' 
if module_path not in sys.path:
    sys.path.append(module_path)

# Now you can import the functions
try:
    from GoogleGemini import generate_video_caption, generate_text_embedding
    print("Successfully imported functions from GoogleGemini.py")
except ImportError as e:
    print(f"Failed to import functions: {e}")
except Exception as e:
    print(f"An error occurred during import: {e}")

# Define your schema
schema = {
    "id": deeplake.types.UInt64(),  # Unique identifier for each entry (e.g., session ID or video clip ID)
    "frames": deeplake.types.Sequence(deeplake.types.Image(sample_compression="jpeg")),  # Video frames as a sequence of images
    "captions": deeplake.types.Text(),  # Single caption for the entire video
    "embeddings": deeplake.types.Embedding(768),  # Embedding for the entire video
}

path = 'al://second-sight/dynamic-video-recordings'

try:
    ds = deeplake.open(path, token = os.getenv("ACTIVELOOP_TOKEN"))
    ds.add_column("frames", deeplake.types.Sequence(deeplake.types.Image(sample_compression="jpeg")))
    ds.add_column("captions", deeplake.types.Text())
    ds.add_column("embeddings", deeplake.types.Embedding(768))
    ds.add_column("id", deeplake.types.UInt64())
    print("Added new columns to the dataset")
    
except Exception as e:
    print(f"Failed to open dataset: {e}")
    ds = deeplake.create(url = path, token= os.getenv("ACTIVELOOP_TOKEN"))
    print(f"Created new dataset at: {path}")


Successfully imported functions from GoogleGemini.py
Added new columns to the dataset


# Successfully Upload to DeepLake

In [ ]:
import deeplake
import time
import os
import cv2
import sys
from dotenv import load_dotenv

caption, embedding = None, None

load_dotenv()
# Add the directory containing GoogleGemini.py to the Python path
# '.' represents the current directory, assuming the notebook is in the same folder as the .py file
module_path = '.' 
if module_path not in sys.path:
    sys.path.append(module_path)

# Now you can import the functions
try:
    from GoogleGemini import generate_video_caption, generate_text_embedding
    print("Successfully imported functions from GoogleGemini.py")
except ImportError as e:
    print(f"Failed to import functions: {e}")
except Exception as e:
    print(f"An error occurred during import: {e}")

# Define your schema
schema = {
    "id": deeplake.types.UInt64(),  # Unique identifier for each entry (e.g., session ID or video clip ID)
    "frames": deeplake.types.Sequence(deeplake.types.Image(sample_compression="jpg")),  # Video frames as a sequence of images
    "captions": deeplake.types.Text(),  # Single caption for the entire video
    "embeddings": deeplake.types.Embedding(768),  # Embedding for the entire video
}

path = 'al://second-sight/video-recordings'

try:
    ds = deeplake.open(path, token = os.getenv("ACTIVELOOP_TOKEN"))
except Exception as e:
    print(f"Failed to open dataset: {e}")
    ds = deeplake.create(url = path,schema=schema, token= os.getenv("ACTIVELOOP_TOKEN"))

# Function to extract frames and motion boxes from video
def extract_frames_and_motion_boxes(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    motion_boxes = []
    previous_frame = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (21, 21), 0)
        current_time = time.time()

        if previous_frame is None:
            previous_frame = gray
        else:
            # Calculate difference between consecutive frames
            delta = cv2.absdiff(previous_frame, gray)
            thresh = cv2.threshold(delta, 50, 255, cv2.THRESH_BINARY)[1]
            thresh = cv2.dilate(thresh, None, iterations=2)
            contours, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            # Detect motion and get bounding boxes
            for c in contours:
                if cv2.contourArea(c) < 1500:  # Ignore small contours
                    continue
                x, y, w, h = cv2.boundingRect(c)
                motion_boxes.append([x, y, w, h])  # Store the bounding box (x, y, width, height)

        frames.append(frame)  # Add frame to list
        previous_frame = gray  # Update previous frame

    cap.release()
    return frames, motion_boxes

# Function to process the video, generate caption, embedding, and upload to DeepLake
async def process_and_upload_video(video_path, dataset):
    if not os.path.exists(video_path):
        print(f"Error: Video file not found at {video_path}")
        return

    print(f"Processing video: {video_path}")
    
    try:
        # 1. Extract Frames and Motion Boxes
        print("Extracting frames and motion boxes...")
        frames, motion_boxes = extract_frames_and_motion_boxes(video_path)
        print(f"Extracted {len(frames)} frames and {len(motion_boxes)} motion boxes.")

        # 2. Generate Caption
        print("Generating caption...")
        caption = generate_video_caption(video_path)
        print(f"Generated Caption: {caption}")

        # 3. Generate Embedding from Caption
        print("Generating embedding...")
        embedding = generate_text_embedding(caption)
        print(f"Generated Embedding (first 5 dims): {embedding[:5]}")
        print(f"Embedding length: {len(embedding)}")

        jpeg_frames = []
        for frame in frames:
            ret, jpeg = cv2.imencode('.jpg', frame)
            if ret:
                jpeg_frames.append(jpeg.tobytes())
        # 4. Prepare data for Deep Lake
        data_to_upload = {
            'id': [int(time.time() * 1000)],  # Unique ID based on timestamp
            'frames': [jpeg_frames],  # Encode frames to JPEG
            'captions': [caption],  # Single caption for the entire video (single text)
            'embeddings': [embedding] ,  # Single embedding for the entire video
        }

        # 5. Append to Deep Lake dataset
        print("Appending data to Deep Lake...")
        dataset.append(data_to_upload)
        print("9. Append successful. Committing...") # ADDED
        dataset.commit()
        print("10. Commit successful.") # ADDED

    except FileNotFoundError:
        print(f"Error: The video file '{video_path}' was not found during processing.")
    except Exception as e:
        print(f"An error occurred during processing or upload: {e}")

# Execute the async function
async def main():
    video_file_path = "tempVideo/FriendsTalking.mov"  # Path to your test video
    print("Starting video processing and upload...")
    await process_and_upload_video(video_file_path, ds)
    print("Process finished.")

# For Jupyter environments, top-level await is allowed
# In other environments, use `asyncio.run(main())` instead
await main()


Successfully imported functions from GoogleGemini.py
Starting video processing and upload...
Processing video: tempVideo/FriendsTalking.mov
Extracting frames and motion boxes...
Extracted 489 frames and 1741 motion boxes.
Generating caption...
Uploading video...
Completed upload: https://generativelanguage.googleapis.com/v1beta/files/dusvtmvej2qq
..............................................................................................Generated Caption: Here's a description of the video:

In this short video clip, we see a casual interaction between two young men in what appears to be a classroom or a similar academic space. The setting is a typical room with a plain ceiling, a whiteboard, and basic furniture. A "MESA" flag hangs on the wall, hinting that the space might belong to a student organization. There's also a microwave, a water cooler, and other supplies visible, further suggesting a shared use area.

One young man is seated at a desk, wearing a black hoodie. He’s relaxed

# Natural Language Query

In [ ]:
import deeplake
from generations import generate_text_embedding
import os
from ConvertVideo import create_video_from_frames, decode_jpeg_frames

# Open the dataset
path = "al://second-sight/video-recordings"  # Replace with your dataset path
ds = deeplake.open(path, token=os.getenv("ACTIVELOOP_TOKEN"))

def query_dataset(ds, query, api_key=os.getenv("TOGETHER_API_KEY")):
    """
    Query the dataset using a text query and return the top N results based on cosine similarity.

    Args:
        ds (deeplake._deeplake.Dataset): The dataset to query
        query (str): The text query to search for
        api_key (str): Together AI API key

    Returns:
        list: A list of dictionaries with video metadata (caption, frames, embedding, and id)
    """
    
    # Step 1: Generate the embedding for the query using the NLP model
    embed_query = generate_text_embedding(query, api_key)
    
    # Convert the embedding list into a string format for SQL query
    str_query = ",".join(str(c) for c in embed_query)

    # Step 2: Query the dataset using cosine similarity
    query_vs = f"""
        SELECT *, cosine_similarity(embeddings, ARRAY[{str_query}]) as score
        ORDER BY cosine_similarity(embeddings, ARRAY[{str_query}]) DESC 
        LIMIT 1  # Adjust this for top N results (e.g., 3)
    """

    # Execute the query
    view_vs = ds.query(query_vs)
    
    # Step 3: Collect and process the results
    data = []
    for row in view_vs:
        test = {
            "frames": row["frames"],  
            "captions": row["captions"],  # Assuming 'captions' is the column storing the caption
            "embeddings": row["embeddings"],  # Assuming 'embedding' contains the video embedding
            "id": row["id"]  # Assuming 'id' is the unique identifier for each video
        }
        print(test)  # You can adjust this to store or process the results as needed
        data.append(test)
    
    return data

# Example usage
query = "Find me a video where someone is delivering a package"
results = query_dataset(ds, query)

output_video_path = "tempVideo/output_video.mp4"  # Desired path for the output video
fps = 30  # Frames per second


# Optionally display or process the results
for result in results:
    frames = decode_jpeg_frames(result['frames'])
    print(f"Caption: {result['captions']}")

    # Call the function to create a video
    if not frames:
        print("Error: No frames provided to create video.")
        break
    print(f"Shape of first frame: {frames[0].shape}")

    success = create_video_from_frames(frames, output_video_path, fps)    
    print(f"Embedding (First 5 Dims): {result['embeddings'][:1]}...")  # Display first 5 dimensions of the embedding
    print(f"ID: {result['id']}")


[SQL-Lexer-Error] Unknown Character: #


{'frames': [array([255, 216, 255, ..., 103, 255, 217], dtype=uint8), array([255, 216, 255, ..., 127, 255, 217], dtype=uint8), array([255, 216, 255, ...,  15, 255, 217], dtype=uint8), array([255, 216, 255, ...,   3, 255, 217], dtype=uint8), array([255, 216, 255, ..., 127, 255, 217], dtype=uint8), array([255, 216, 255, ...,   7, 255, 217], dtype=uint8), array([255, 216, 255, ...,   1, 255, 217], dtype=uint8), array([255, 216, 255, ...,   7, 255, 217], dtype=uint8), array([255, 216, 255, ...,  15, 255, 217], dtype=uint8), array([255, 216, 255, ...,  31, 255, 217], dtype=uint8), array([255, 216, 255, ...,   0, 255, 217], dtype=uint8), array([255, 216, 255, ...,   0, 255, 217], dtype=uint8), array([255, 216, 255, ...,   3, 255, 217], dtype=uint8), array([255, 216, 255, ...,  15, 255, 217], dtype=uint8), array([255, 216, 255, ...,   3, 255, 217], dtype=uint8), array([255, 216, 255, ...,  39, 255, 217], dtype=uint8), array([255, 216, 255, ..., 127, 255, 217], dtype=uint8), array([255, 216, 25

# Test JPEG

In [20]:
import deeplake
import time
import os
import cv2
import sys
from dotenv import load_dotenv

caption, embedding = None, None

load_dotenv()
# Add the directory containing GoogleGemini.py to the Python path
# '.' represents the current directory, assuming the notebook is in the same folder as the .py file
module_path = '.' 
if module_path not in sys.path:
    sys.path.append(module_path)

# Now you can import the functions
try:
    from GoogleGemini import generate_video_caption, generate_text_embedding
    print("Successfully imported functions from GoogleGemini.py")
except ImportError as e:
    print(f"Failed to import functions: {e}")
except Exception as e:
    print(f"An error occurred during import: {e}")

# Define your schema
schema = {
    "id": deeplake.types.UInt64(),  # Unique identifier for each entry (e.g., session ID or video clip ID)
    "frames": deeplake.types.Sequence(deeplake.types.Image(sample_compression="jpeg")),  # Video frames as a sequence of images
    "captions": deeplake.types.Text(),  # Single caption for the entire video
    "embeddings": deeplake.types.Embedding(768),  # Embedding for the entire video
}

path = 'al://second-sight/test-jpeg'

try:
    ds = deeplake.open(path, token = os.getenv("ACTIVELOOP_TOKEN"))
except Exception as e:
    print(f"Failed to open dataset: {e}")
    ds = deeplake.create(url = path,schema=schema, token= os.getenv("ACTIVELOOP_TOKEN"))

# Function to extract frames and motion boxes from video
def extract_frames_and_motion_boxes(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    motion_boxes = []
    previous_frame = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (21, 21), 0)
        current_time = time.time()

        if previous_frame is None:
            previous_frame = gray
        else:
            # Calculate difference between consecutive frames
            delta = cv2.absdiff(previous_frame, gray)
            thresh = cv2.threshold(delta, 50, 255, cv2.THRESH_BINARY)[1]
            thresh = cv2.dilate(thresh, None, iterations=2)
            contours, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            # Detect motion and get bounding boxes
            for c in contours:
                if cv2.contourArea(c) < 1500:  # Ignore small contours
                    continue
                x, y, w, h = cv2.boundingRect(c)
                motion_boxes.append([x, y, w, h])  # Store the bounding box (x, y, width, height)

        frames.append(frame)  # Add frame to list
        previous_frame = gray  # Update previous frame

    cap.release()
    return frames, motion_boxes

# Function to process the video, generate caption, embedding, and upload to DeepLake
async def process_and_upload_video(video_path, dataset):
    if not os.path.exists(video_path):
        print(f"Error: Video file not found at {video_path}")
        return

    print(f"Processing video: {video_path}")
    
    try:
        # 1. Extract Frames and Motion Boxes
        print("Extracting frames and motion boxes...")
        frames, motion_boxes = extract_frames_and_motion_boxes(video_path)
        print(f"Extracted {len(frames)} frames and {len(motion_boxes)} motion boxes.")

        # 2. Generate Caption
        print("Generating caption...")
        caption = generate_video_caption(video_path)
        print(f"Generated Caption: {caption}")

        # 3. Generate Embedding from Caption
        print("Generating embedding...")
        embedding = generate_text_embedding(caption)
        print(f"Generated Embedding (first 5 dims): {embedding[:5]}")
        print(f"Embedding length: {len(embedding)}")

        jpeg_frames = []
        for frame in frames:
            ret, jpeg = cv2.imencode('.jpeg', frame)
            if ret:
                jpeg_frames.append(jpeg.tobytes())
        # 4. Prepare data for Deep Lake
        data_to_upload = {
            'id': [int(time.time() * 1000)],  # Unique ID based on timestamp
            'frames': [jpeg_frames],  # Encode frames to JPEG
            'captions': [caption],  # Single caption for the entire video (single text)
            'embeddings': [embedding] ,  # Single embedding for the entire video
        }

        # 5. Append to Deep Lake dataset
        print("Appending data to Deep Lake...")
        dataset.append(data_to_upload)
        print("9. Append successful. Committing...") # ADDED
        dataset.commit()
        print("10. Commit successful.") # ADDED

    except FileNotFoundError:
        print(f"Error: The video file '{video_path}' was not found during processing.")
    except Exception as e:
        print(f"An error occurred during processing or upload: {e}")

# Execute the async function
async def main():
    video_file_path = "tempVideo/testdelivery.mov"  # Path to your test video
    print("Starting video processing and upload...")
    await process_and_upload_video(video_file_path, ds)
    print("Process finished.")

# For Jupyter environments, top-level await is allowed
# In other environments, use `asyncio.run(main())` instead
await main()


Successfully imported functions from GoogleGemini.py
Starting video processing and upload...
Processing video: tempVideo/testdelivery.mov
Extracting frames and motion boxes...
Extracted 903 frames and 695 motion boxes.
Generating caption...
Uploading video...
Completed upload: https://generativelanguage.googleapis.com/v1beta/files/suj342ldg1b2
....................Generated Caption: Here's a breakdown of the video:

**Main Action/Event:** The video shows a delivery person acting out a humorous scenario where they prank the homeowner. They place a delivery box (likely from Amazon, judging by the logo) in front of the door, and then quickly grab an orange spray can. They mime spraying the package or perhaps the homeowner. They then check their phone and leave.

**Setting and Environment:** The scene takes place outside a modern home. The backdrop is a dark, paneled front door with a decorative glass panel, framed by white brick pillars. The flooring appears to be a light-colored tile. The

In [21]:
import deeplake
from generations import generate_text_embedding
import os
from ConvertVideo import create_video_from_frames
# Open the dataset
path = "al://second-sight/test-jpeg"  # Replace with your dataset path
ds = deeplake.open(path, token=os.getenv("ACTIVELOOP_TOKEN"))

def query_dataset(ds, query, api_key=os.getenv("TOGETHER_API_KEY")):
    """
    Query the dataset using a text query and return the top N results based on cosine similarity.

    Args:
        ds (deeplake._deeplake.Dataset): The dataset to query
        query (str): The text query to search for
        api_key (str): Together AI API key

    Returns:
        list: A list of dictionaries with video metadata (caption, frames, embedding, and id)
    """
    
    # Step 1: Generate the embedding for the query using the NLP model
    embed_query = generate_text_embedding(query, api_key)
    
    # Convert the embedding list into a string format for SQL query
    str_query = ",".join(str(c) for c in embed_query)

    # Step 2: Query the dataset using cosine similarity
    query_vs = f"""
        SELECT *, cosine_similarity(embeddings, ARRAY[{str_query}]) as score
        ORDER BY cosine_similarity(embeddings, ARRAY[{str_query}]) DESC 
        LIMIT 3  # Adjust this for top N results (e.g., 3)
    """

    # Execute the query
    view_vs = ds.query(query_vs)
    
    # Step 3: Collect and process the results
    data = []
    for row in view_vs:
        test = {
            "frames": row["frames"],  
            "captions": row["captions"],  # Assuming 'captions' is the column storing the caption
            "embeddings": row["embeddings"],  # Assuming 'embedding' contains the video embedding
            "id": row["id"]  # Assuming 'id' is the unique identifier for each video
        }
        print(test)  # You can adjust this to store or process the results as needed
        data.append(test)
    
    return data

# Example usage
query = "Find me a video where friends are talking"
results = query_dataset(ds, query)

output_video_path = "tempVideo/output_video.mp4"  # Desired path for the output video
fps = 45  # Frames per second


# Optionally display or process the results
for result in results:
    frames = result['frames']
    # Call the function to create a video
    if not frames:
        print("Error: No frames provided to create video.")
        break
    print(f"Shape of first frame: {frames[0].shape}")

    success = create_video_from_frames(result['frames'], output_video_path, fps)    
    print(f"Caption: {result['captions']}")
    print(f"Embedding (First 5 Dims): {result['embeddings'][:5]}...")  # Display first 5 dimensions of the embedding
    print(f"ID: {result['id']}")


[SQL-Lexer-Error] Unknown Character: #


{'frames': [array([255, 216, 255, ..., 103, 255, 217], dtype=uint8), array([255, 216, 255, ..., 127, 255, 217], dtype=uint8), array([255, 216, 255, ...,  15, 255, 217], dtype=uint8), array([255, 216, 255, ...,   3, 255, 217], dtype=uint8), array([255, 216, 255, ..., 127, 255, 217], dtype=uint8), array([255, 216, 255, ...,   7, 255, 217], dtype=uint8), array([255, 216, 255, ...,   1, 255, 217], dtype=uint8), array([255, 216, 255, ...,   7, 255, 217], dtype=uint8), array([255, 216, 255, ...,  15, 255, 217], dtype=uint8), array([255, 216, 255, ...,  31, 255, 217], dtype=uint8), array([255, 216, 255, ...,   0, 255, 217], dtype=uint8), array([255, 216, 255, ...,   0, 255, 217], dtype=uint8), array([255, 216, 255, ...,   3, 255, 217], dtype=uint8), array([255, 216, 255, ...,  15, 255, 217], dtype=uint8), array([255, 216, 255, ...,   3, 255, 217], dtype=uint8), array([255, 216, 255, ...,  39, 255, 217], dtype=uint8), array([255, 216, 255, ..., 127, 255, 217], dtype=uint8), array([255, 216, 25

ValueError: not enough values to unpack (expected 3, got 1)

In [ ]:
ds = deeplake.open(path)
def query_dataset(ds, query, api_key=os.getenv("TOGETHER_API_KEY")):
    """
    Query the dataset using a text query and return the top 3 results based on cosine similarity.

    Args:
        ds (deeplake._deeplake.Dataset): The dataset to query
        query (str): The text query to search for
        api_key (str): Together AI API key

    Returns:
        deeplake._deeplake.DatasetView: The top 3 results from the query
    """
    
    
    embed_query = generate_text_embedding(query, api_key)
    str_query = ",".join(str(c) for c in embed_query)

    query_vs = f"""
        SELECT *, cosine_similarity(embedding, ARRAY[{str_query}]) as score
        FROM (
            SELECT *, ROW_NUMBER() AS row_id
        )
        ORDER BY cosine_similarity(embedding, ARRAY[{str_query}]) DESC 

        LIMIT 1
    """

    view_vs = ds.query(query_vs)
    data = []
    for row in view_vs:
        test = {
            "caption": row["caption"],
            "frames": row["frames"],
            "embedding": row["embedding"]
        }
        print(test)
        data.append(test)
    return data


In [3]:
#!/usr/bin/env python3
"""
test_firestore_connection.py

A quick script to verify that your Python backend can connect to Firestore,
write a test document, read it back, and clean up.
"""

import os
import sys
import uuid
import logging
from dotenv import load_dotenv
import firebase_admin
from firebase_admin import credentials, firestore

# 1) (Optional) Load environment variables from .env, if you store your key path there
load_dotenv()

# Path to your Firebase service account JSON
SERVICE_ACCOUNT_PATH = os.getenv("FIREBASE_SERVICE_ACCOUNT", "mmh2025-f6143-firebase-adminsdk-fbsvc-02a58364e6.json")

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger()

def main():
    # 2) Initialize Firebase Admin SDK
    try:
        cred = credentials.Certificate(SERVICE_ACCOUNT_PATH)
        firebase_admin.initialize_app(cred)
        log.info("✅ Firebase Admin initialized successfully")
    except Exception as e:
        log.error(f"❌ Failed to initialize Firebase Admin: {e}")
        sys.exit(1)

    # 3) Create Firestore client
    try:
        db = firestore.client()
        log.info(f"✅ Firestore client created (project: {db.project})")
    except Exception as e:
        log.error(f"❌ Failed to create Firestore client: {e}")
        sys.exit(1)

    # 4) Write a test document
    test_collection = "testConnection"
    test_doc_id = str(uuid.uuid4())
    test_data = {"ping": "pong", "timestamp": firestore.SERVER_TIMESTAMP}

    try:
        db.collection(test_collection).document(test_doc_id).set(test_data)
        log.info(f"✅ Wrote test document '{test_doc_id}' to '{test_collection}/{test_doc_id}'")
    except Exception as e:
        log.error(f"❌ Failed to write test document: {e}")
        sys.exit(1)

    # 5) Read it back
    try:
        snapshot = db.collection(test_collection).document(test_doc_id).get()
        if snapshot.exists and snapshot.to_dict().get("ping") == "pong":
            log.info(f"✅ Read back test document: {snapshot.to_dict()}")
        else:
            log.error(f"❌ Document read but unexpected data: {snapshot.to_dict()}")
            sys.exit(1)
    except Exception as e:
        log.error(f"❌ Failed to read test document: {e}")
        sys.exit(1)

    # 6) Clean up
    try:
        db.collection(test_collection).document(test_doc_id).delete()
        log.info("✅ Cleaned up test document")
    except Exception as e:
        log.warning(f"⚠️ Failed to delete test document: {e}")

if __name__ == "__main__":
    main()


✅ Firebase Admin initialized successfully
✅ Firestore client created (project: mmh2025-f6143)
✅ Wrote test document 'd83bca1d-befc-45c0-925a-f4eb2b3cfb28' to 'testConnection/d83bca1d-befc-45c0-925a-f4eb2b3cfb28'
✅ Read back test document: {'timestamp': DatetimeWithNanoseconds(2025, 4, 28, 6, 57, 6, 469000, tzinfo=datetime.timezone.utc), 'ping': 'pong'}
✅ Cleaned up test document
